In [5]:
import torch.nn

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.depth = d_model // num_heads
        
        self.wq = torch.nn.Linear(d_model, d_model)
        self.wk = torch.nn.Linear(d_model, d_model)
        self.wv = torch.nn.Linear(d_model, d_model)
        
        self.dense = torch.nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.permute(0, 2, 1, 3)
    
    def forward(self, v, k, q, mask):
        batch_size = q.size(0)
        
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)
        
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        scaled_attention, _ = self.scaled_dot_product_attention(q, k, v, mask)
        
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()
        scaled_attention = scaled_attention.view(batch_size, -1, self.d_model)
        
        output = self.dense(scaled_attention)
        
        return output
    def scaled_dot_product_attention(self, q, k, v, mask):
        matmul_qk = torch.matmul(q, k.transpose(-2, -1))
        
        dk = k.size(-1)
        scaled_attention_logits = matmul_qk / torch.sqrt(torch.tensor(dk, dtype=torch.float32))
        
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)
        
        attention_weights = torch.nn.Softmax(dim=-1)(scaled_attention_logits)
        
        output = torch.matmul(attention_weights, v)
        
        return output, attention_weights
# Example usage
d_model = 512
num_heads = 8
mha = MultiHeadAttention(d_model, num_heads)
q = torch.rand(64, 10, d_model)  # (batch_size,seq_len_q, d_model)
k = torch.rand(64, 10, d_model)  # (batch_size, seq_len_k, d_model)
v = torch.rand(64, 10, d_model)  # (batch_size, seq_len_v, d_model)
mask = None  # Optional mask
output = mha(v, k, q, mask)
print(output.shape)  # Expected output: (batch_size, seq_len_q, d_model)    


torch.Size([64, 10, 512])


In [7]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(EncoderBlock, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model,dff),
            torch.nn.ReLU(),
            torch.nn.Linear(dff,d_model)
        )
        self.layernorm1 = torch.nn.LayerNorm(d_model)
        self.layernorm2 = torch.nn.LayerNorm(d_model)
        self.dropout1 = torch.nn.Dropout(dropout_rate)
        self.dropout2 = torch.nn.Dropout(dropout_rate)
        

    def forward(self, x, mask):
        attn_output = self.mha(x, x, x, mask)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(x + attn_output)
        
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        out2 = self.layernorm2(out1 + ffn_output)
        
        return out2

# Example usage
d_model = 512   
num_heads = 8
dff = 2048
encoder_block = EncoderBlock(d_model, num_heads, dff)
x = torch.rand(64, 10, d_model)  # (batch_size, seq_len, d_model)
mask = None  # Optional mask
output = encoder_block(x, mask)
print(output.shape)  # Expected output: (batch_size, seq_len, d_model)  

torch.Size([64, 10, 512])


In [8]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
        super(DecoderBlock, self).__init__()
        self.mha1 = MultiHeadAttention(d_model, num_heads)
        self.mha2 = MultiHeadAttention(d_model, num_heads)
        
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model,dff),
            torch.nn.ReLU(),
            torch.nn.Linear(dff,d_model)
        )
        
        self.layernorm1 = torch.nn.LayerNorm(d_model)
        self.layernorm2 = torch.nn.LayerNorm(d_model)
        self.layernorm3 = torch.nn.LayerNorm(d_model)
        
        self.dropout1 = torch.nn.Dropout(dropout_rate)
        self.dropout2 = torch.nn.Dropout(dropout_rate)
        self.dropout3 = torch.nn.Dropout(dropout_rate)

    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        attn1 = self.mha1(x, x, x, look_ahead_mask)
        attn1 = self.dropout1(attn1)
        out1 = self.layernorm1(attn1 + x)
        
        attn2 = self.mha2(enc_output, enc_output, out1, padding_mask)
        attn2 = self.dropout2(attn2)
        out2 = self.layernorm2(attn2 + out1)
        
        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output)
        out3 = self.layernorm3(ffn_output + out2)
        
        return out3

# Example usage
d_model = 512
num_heads = 8
dff = 2048
decoder_block = DecoderBlock(d_model, num_heads, dff)
x = torch.rand(64, 10, d_model)  # (batch_size, target_seq_len, d_model)
enc_output = torch.rand(64, 10, d_model)  # (batch_size, input_seq_len, d_model)
look_ahead_mask = None  # Optional look-ahead mask  
padding_mask = None  # Optional padding mask
output = decoder_block(x, enc_output, look_ahead_mask, padding_mask)    
print(output.shape)  # Expected output: (batch_size, seq_len, d_model)  

torch.Size([64, 10, 512])


In [10]:
!pip install datasets

Looking in indexes: https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
  Using cached https://mirrors.tuna.tsinghua.edu.cn/pypi/web/packages/1e/03/c6d9c3119cf712f638fe763e887ecaac6acbb62bf1e2acc3cbde0df340fd/datasets-4.7.0-py3-none-any.whl (527 kB)
  Using cached https://mirrors.tuna.tsinghua.edu.cn/pypi/web/packages/d5/09/a532297c9591a727d67760e2e756b83905dd89adb365a7f6e9c72578bcc1/pyarrow-23.0.1-cp313-cp313-win_amd64.whl (27.5 MB)
  Using cached https://mirrors.tuna.tsinghua.edu.cn/pypi/web/packages/50/3d/9373ad9c56321fdab5b41197068e1d8c25883b3fea29dd361f9b55116869/dill-0.4.0-py3-none-any.whl (119 kB)
     ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
     - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
     ------ --------------------------------- 1.6/9.7 MB 4.7 MB/s eta 0:00:02
     ------------- -------------------------- 3.4/9.7 MB 6.3 MB/s eta 0:00:01
     -------------------------- ------------- 6.6/9.7 MB 9.0 MB/s eta 0:00:01
     ------

In [2]:
import os
import requests
from datasets import Dataset

# 1. 设置镜像 (可选，因为 raw.githubusercontent.com 通常国内也能访问，如果慢也可以找镜像)
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com" 

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
local_file_path = "data/shakespeare.txt"

# 创建 data 目录
os.makedirs("data", exist_ok=True)

print("正在下载莎士比亚原始数据...")
if not os.path.exists(local_file_path):
    response = requests.get(url)
    if response.status_code == 200:
        with open(local_file_path, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"✅ 下载成功，保存至: {local_file_path}")
    else:
        raise Exception("下载失败，请检查网络")
else:
    print("✅ 文件已存在，跳过下载")

# 2. 读取本地文本
with open(local_file_path, "r", encoding="utf-8") as f:
    text_data = f.read()

# 3. 转换为 HuggingFace Dataset 格式
# 我们手动构建一个字典列表，这是最通用的格式
data_dict = {"text": [text_data]} 
# 注意：这里我们把整本书作为一个样本。
# 如果你想按行分割，可以用 text_data.split('\n')，但那样会切断句子上下文。
# 对于字符级 LM，通常把整个长文本作为一个样本，然后在 DataLoader 里做滑动窗口切割。

dataset = Dataset.from_dict(data_dict)

print("-" * 30)
print(f"✅ 数据集构建成功！")
print(f"样本数: {len(dataset)}")
print(f"总字符数: {len(dataset[0]['text']):,}")
print(f"前 100 字符预览:\n{dataset[0]['text'][:100]}")

正在下载莎士比亚原始数据...
✅ 下载成功，保存至: data/shakespeare.txt
------------------------------
✅ 数据集构建成功！
样本数: 1
总字符数: 1,115,394
前 100 字符预览:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
import os

class ShakespeareCharDataset(Dataset):
    def __init__(self, text, block_size, train=True, val_fraction=0.05, test_fraction=0.05):
        """
        text: 完整的莎士比亚文本字符串
        block_size: 上下文窗口长度 (例如 64 或 128)
        train: 是否处于训练模式 (决定使用哪部分数据)
        """
        self.block_size = block_size
        
        # 1. 构建词汇表 (Character Level)
        # sorted 保证每次运行顺序一致，确保复现性
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        self.stoi = {ch: i for i, ch in enumerate(chars)}
        self.itos = {i: ch for i, ch in enumerate(chars)}
        
        # 2. 数字化整个文本
        self.data_ids = [self.stoi[c] for c in text]
        total_len = len(self.data_ids)
        
        # 3. 按顺序划分数据集 (关键！防止数据泄露)
        # 假设: Train 90%, Val 5%, Test 5%
        train_end = int(total_len * (1 - val_fraction - test_fraction))
        val_end = int(total_len * (1 - test_fraction))
        
        if train:
            # 训练集：取前 90%
            self.data_ids = self.data_ids[:train_end]
            self.mode = "Train"
        else:
            # 这里为了简单，我们把 Val 和 Test 合并为 "Eval" 模式
            # 实际工程中通常分开实例化两个 Dataset
            # 这里我们取最后 10% 作为评估集
            self.data_ids = self.data_ids[train_end:]
            self.mode = "Eval"
            
        self.length = len(self.data_ids)
        print(f"[{self.mode}] 数据加载完成: 字符数 {self.length:,}, 词汇表大小 {self.vocab_size}")

    def __len__(self):
        # 滑动窗口能生成的样本数 = 总长度 - block_size
        # 如果数据太短连一个 block 都凑不够，返回 0
        return max(0, self.length - self.block_size)

    def __getitem__(self, idx):
        # 截取输入 x 和 目标 y (y 是 x 向后移一位)
        # x: [idx, idx+1, ..., idx+block_size-1]
        # y: [idx+1, idx+2, ..., idx+block_size]
        chunk = self.data_ids[idx : idx + self.block_size + 1]
        
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        
        return x, y

    def decode(self, ids):
        return ''.join([self.itos[i] for i in ids])

    def encode(self, text):
        return [self.stoi[c] for c in text]

# --- 使用示例 ---
if __name__ == "__main__":
    # 模拟读取数据 (实际请替换为你之前的下载代码)
    with open("data/shakespeare.txt", "r", encoding="utf-8") as f:
        full_text = f.read()
    
    BLOCK_SIZE = 64
    BATCH_SIZE = 32
    
    # 实例化训练集
    train_dataset = ShakespeareCharDataset(full_text, BLOCK_SIZE, train=True)
    
    # 实例化验证/测试集
    eval_dataset = ShakespeareCharDataset(full_text, BLOCK_SIZE, train=False)
    
    # 构建 DataLoader
    # num_workers=0 在 Windows 上最稳定，Linux/Mac 可以设为 CPU 核心数加速
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # 验证一下
    print("\n--- 采样检查 ---")
    xb, yb = next(iter(train_loader))
    print(f"Input Shape: {xb.shape}, Target Shape: {yb.shape}")
    print(f"第一句输入 (解码): {train_dataset.decode(xb[0].tolist())}")
    print(f"第一句目标 (解码): {train_dataset.decode(yb[0].tolist())}")
    print("-" * 30)
    print("✅ 数据集准备就绪，可以开始训练模型了！")

[Train] 数据加载完成: 字符数 1,003,854, 词汇表大小 65
[Eval] 数据加载完成: 字符数 111,540, 词汇表大小 65

--- 采样检查 ---
Input Shape: torch.Size([32, 64]), Target Shape: torch.Size([32, 64])
第一句输入 (解码): hing else so happy
As in a soul remembering my good friends;
And
第一句目标 (解码): ing else so happy
As in a soul remembering my good friends;
And,
------------------------------
✅ 数据集准备就绪，可以开始训练模型了！


In [ ]:
import os
import time
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from contextlib import nullcontext

# =================配置区域=================
class TrainConfig:
    # 数据路径
    data_path = "data/shakespeare.txt"
    
    # 模型超参数 (与 dataset 中的 block_size 保持一致)
    block_size = 64
    vocab_size = 65  # 会在代码中动态校验
    n_layer = 6      # 稍微加深一点，6层效果不错
    n_head = 6
    n_embd = 192
    dropout = 0.2    # 加入 Dropout 防止过拟合
    
    # 训练超参数
    batch_size = 64
    max_iters = 5000       # 总训练步数
    eval_interval = 500    # 每隔多少步验证一次
    eval_iters = 200       # 验证时跑多少个 batch
    learning_rate = 3e-4
    weight_decay = 1e-1
    beta1 = 0.9
    beta2 = 0.99
    grad_clip = 1.0        # 梯度裁剪阈值
    
    # 设备配置
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16'
    
    # 保存路径
    out_dir = 'out'
    always_save_checkpoint = True

cfg = TrainConfig()


# =================辅助函数=================
def estimate_loss(model, train_loader, eval_loader, ctx):
    model.eval()
    losses = torch.zeros(eval_iters)
    
    # 轮流从训练集和验证集采样评估，或者直接只用验证集
    # 这里为了简单，我们只评估验证集 (eval_loader)
    loader = eval_loader 
    
    with ctx:
        for k in range(eval_iters):
            xb, yb = next(iter(loader))
            xb, yb = xb.to(cfg.device), yb.to(cfg.device)
            with torch.no_grad():
                logits, loss = model(xb, yb)
            losses[k] = loss.item()
    
    model.train()
    return losses.mean()



# =================主训练流程=================
def main():
    print(f"🚀 正在初始化训练... 设备: {cfg.device}, 精度: {cfg.dtype}")
    
    # 1. 准备数据
    print("📂 加载数据集...")
    with open(cfg.data_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    train_dataset = ShakespeareCharDataset(text, cfg.block_size, train=True)
    eval_dataset = ShakespeareCharDataset(text, cfg.block_size, train=False)
    
    # 校验词汇表大小
    if train_dataset.vocab_size != cfg.vocab_size:
        print(f"⚠️ 警告: 配置的 vocab_size ({cfg.vocab_size}) 与实际数据 ({train_dataset.vocab_size}) 不符，自动修正。")
        cfg.vocab_size = train_dataset.vocab_size
    
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
    eval_loader = DataLoader(eval_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    
    # 创建一个迭代器生成器，避免每个 epoch 重新 shuffle 导致逻辑复杂
    # 这里采用简单的无限迭代器方式
    def get_batch(loader):
        while True:
            for xb, yb in loader:
                yield xb.to(cfg.device), yb.to(cfg.device)
    
    train_iter = get_batch(train_loader)
    eval_iter = get_batch(eval_loader)

    # 2. 初始化模型
    print("🏗️ 构建模型...")
    model = ShakespeareTransformer(Config(
        block_size=cfg.block_size,
        vocab_size=cfg.vocab_size,
        n_layer=cfg.n_layer,
        n_head=cfg.n_head,
        n_embd=cfg.n_embd,
        dropout=cfg.dropout
    )).to(cfg.device)
    
    # 统计参数量
    total_params = sum(p.numel() for p in model.parameters())
    print(f"📊 模型总参数量: {total_params:,} ({total_params/1e6:.2f} M)")

    # 3. 优化器与学习率调度
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=cfg.learning_rate, 
        betas=(cfg.beta1, cfg.beta2), 
        weight_decay=cfg.weight_decay,
        fused=True if cfg.device == 'cuda' else False # CUDA 下开启融合加速
    )
    
    # 简单的线性 Warmup + 余弦退火 (简化版：直接用 Constant 或 Step)
    # 为了代码简洁，这里先用 Constant，进阶可加 Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.max_iters, eta_min=1e-5)

    # 4. 混合精度训练设置
    ctx = nullcontext() if cfg.device == 'cpu' else autocast(device_type=cfg.device, dtype=getattr(torch, cfg.dtype))
    scaler = GradScaler(enabled=(cfg.device == 'cuda' and cfg.dtype == 'float16'))

    # 5. 训练循环
    os.makedirs(cfg.out_dir, exist_ok=True)
    best_val_loss = float('inf')
    
    print(f"🔥 开始训练 (Max Iters: {cfg.max_iters})...")
    t0 = time.time()
    local_iter_num = 0
    
    model.train()
    
    for step in range(cfg.max_iters):
        # 获取一个 Batch
        xb, yb = next(train_iter)
        
        # 学习率预热 (可选，前 100 步线性增加)
        if step < 100:
            lr_scale = step / 100.0
            for param_group in optimizer.param_groups:
                param_group['lr'] = cfg.learning_rate * lr_scale

        with ctx:
            logits, loss = model(xb, yb)
            loss = loss / 1 # 梯度累积步骤数为1时的写法
            
        # 反向传播
        optimizer.zero_grad(set_to_none=True)
        
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()
            
        scheduler.step()
        
        # 日志与验证
        if step % cfg.eval_interval == 0 or step == cfg.max_iters - 1:
            val_loss = estimate_loss(model, train_loader, eval_loader, ctx)
            dt = time.time() - t0
            print(f"Step {step}: train_loss={loss.item():.4f}, val_loss={val_loss:.4f}, time={dt:.2f}s")
            
            # 保存最佳模型
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                checkpoint = {
                    'model': model.state_dict(),
                    'config': cfg,
                    'val_loss': val_loss,
                    'step': step
                }
                save_path = os.path.join(cfg.out_dir, 'best_model.pt')
                torch.save(checkpoint, save_path)
                print(f"💾 保存最佳模型至: {save_path}")
        
        # 进度条简易版
        if step % 100 == 0:
            local_iter_num += 1
            
    print("✅ 训练完成！")

main()

🚀 正在初始化训练... 设备: cpu, 精度: float16
📂 加载数据集...
[Train] 数据加载完成: 字符数 1,003,854, 词汇表大小 65
[Eval] 数据加载完成: 字符数 111,540, 词汇表大小 65
🏗️ 构建模型...


NameError: name 'ShakespeareTransformer' is not defined

In [ ]:
# 6. 生成测试 (训练完立刻看看效果)
    print("\n🎨 正在生成样本...")
    model.eval()
    start_ids = train_dataset.encode("ROMEO: ")
    x = torch.tensor([start_ids], dtype=torch.long, device=cfg.device)
    
    generated_ids = x
    with torch.no_grad():
        for _ in range(200):
            x_cond = x if x.size(1) <= cfg.block_size else x[:, -cfg.block_size:]
            logits, _ = model(x_cond)
            logits = logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1, temperature=0.8)
            x = torch.cat((x, idx_next), dim=1)
            generated_ids = x
            
    generated_text = train_dataset.decode(generated_ids[0].tolist())
    print("-" * 30)
    print(generated_text)
    print("-" * 30)

if __name__ == "__main__":
    main()